# Chapter 3: Google Maps
*System Design Interview Volume 2*

## TL;DR

Google Maps supports location updates, navigation with ETA, and map rendering for 1B DAU. Map rendering uses **pre-generated tiles at 21 zoom levels** served via CDN (~100 PB total). Navigation relies on **routing tiles** -- road graphs partitioned into small chunks at three detail levels (local, arterial, highway) -- consumed by **A* pathfinding** on demand. Location updates stream through **Kafka** to feed traffic analysis, routing tile updates, and ML-based ETA prediction. The key insight: **decompose the world into tiles** for both rendering and routing.

## Requirements

| Requirement | Detail |
|---|---|
| Location update | Record user GPS positions during navigation |
| Navigation + ETA | Find fast routes with accurate time estimates |
| Map rendering | Smooth, low-bandwidth map display on mobile |
| Accuracy | Never give wrong directions |
| Low data usage | Minimize mobile bandwidth and battery consumption |
| Multi-mode | Support driving, walking, bus, etc. |

## Estimation: Storage and QPS

In [ ]:
# Back-of-the-envelope: Google Maps
dau = 1_000_000_000        # 1B DAU
nav_minutes_per_week = 35   # per user
nav_minutes_per_day = nav_minutes_per_week / 7

total_nav_minutes_per_day = dau * nav_minutes_per_day
print(f"Total navigation minutes/day: {total_nav_minutes_per_day/1e9:.1f} billion")

# Location update QPS (batched every 15 seconds)
gps_updates_per_day = total_nav_minutes_per_day * 60 / 15  # one batch per 15s
avg_qps = gps_updates_per_day / 86_400
peak_qps = avg_qps * 5
print(f"Location update avg QPS: {avg_qps:,.0f}")
print(f"Location update peak QPS: {peak_qps:,.0f}")

# Map tile storage
print(f"\n--- Map Tile Storage ---")
tiles_at_zoom_21 = 4.4e12   # 4.4 trillion tiles
tile_size_kb = 100           # 256x256 PNG
raw_pb = tiles_at_zoom_21 * tile_size_kb / (1024**4)  # convert KB to PB
print(f"Raw storage at zoom 21: {raw_pb:.0f} PB")
compressed_pb = raw_pb * 0.12  # 88% compression (oceans, deserts)
print(f"After compression (90% uninhabited): {compressed_pb:.0f} PB")

# Total across all zoom levels: geometric series sum ~= 4/3 * highest level
total_pb = compressed_pb * (4/3)
print(f"Total across all zoom levels: {total_pb:.0f} PB")

# Data usage per user during navigation
speed_kmh = 30
tile_coverage_m = 200        # each tile covers 200m x 200m
tiles_per_km2 = (1000 / tile_coverage_m) ** 2
tile_size_mb = tile_size_kb / 1024
data_per_km2_mb = tiles_per_km2 * tile_size_mb
data_per_hour_mb = speed_kmh * data_per_km2_mb / tiles_per_km2 * tiles_per_km2
print(f"\nData usage per hour at {speed_kmh} km/h: {data_per_hour_mb:.1f} MB")
print(f"Data usage per minute: {data_per_hour_mb / 60:.2f} MB")

## High-Level Design

```mermaid
graph TB
    User[Mobile User]
    LB[Load Balancer]
    LS[Location Service]
    NS[Navigation Service]
    CDN[CDN<br>Map Tile POPs]
    MTS[Map Tile Service]

    GDB[(Geocoding DB<br>Redis)]
    RT[(Routing Tiles<br>S3 Object Storage)]
    UL[(User Location DB<br>Cassandra)]
    ORIGIN[(Map Tile Origin<br>S3)]

    User --> LB
    User -->|fetch tiles| CDN
    CDN -->|cache miss| ORIGIN
    LB --> LS
    LB --> NS
    LB --> MTS
    LS --> UL
    NS --> GDB
    NS --> RT
    MTS --> ORIGIN

    style LS fill:#b3e6b3,stroke:#333
    style NS fill:#ffcc99,stroke:#333
    style CDN fill:#cce5ff,stroke:#333
```

**Three core services:**
1. **Location Service** -- ingests GPS batches, writes to Cassandra, streams to Kafka
2. **Navigation Service** -- route planning with shortest-path, ETA, and ranking
3. **Map Tile Service** -- resolves location + zoom to tile URLs from CDN

## Deep Dive: Map Tiling

```mermaid
graph LR
    subgraph Z0["Zoom 0: 1 tile"]
        T0["256x256 = world"]
    end
    subgraph Z1["Zoom 1: 4 tiles"]
        T1a["NW"] --- T1b["NE"]
        T1c["SW"] --- T1d["SE"]
    end
    subgraph Z2["Zoom 2: 16 tiles"]
        T2["4x4 grid"]
    end
    Z0 -->|2x each axis| Z1
    Z1 -->|2x each axis| Z2
```

- **21 zoom levels**: each level has 4x tiles of the previous
- Tiles are **static PNGs** identified by geohash, served from CDN
- Client computes tile URL from `(lat, lng, zoom)` --> geohash --> `cdn.example.com/tiles/{geohash}.png`
- **Vector tiles** (WebGL) compress better and zoom more smoothly than raster

## Deep Dive: Routing Tiles and Navigation

```mermaid
graph TB
    subgraph RP["Route Planner"]
        SP[Shortest-Path Service<br>A* on routing tiles]
        ETA[ETA Service<br>ML traffic prediction]
        RK[Ranker<br>filters + ranking]
    end

    GEO[Geocoding Service] -->|lat,lng| SP
    SP -->|top-k routes| ETA
    ETA -->|time estimates| RK
    RK -->|ranked routes| User[Client]

    subgraph RT["Routing Tiles (3 levels)"]
        L1["Level 1: Local streets<br>small tiles"]
        L2["Level 2: Arterial roads<br>medium tiles"]
        L3["Level 3: Highways<br>large tiles"]
        L1 ---|cross-level edges| L2
        L2 ---|cross-level edges| L3
    end

    SP -->|load on demand| RT

    style SP fill:#b3e6b3,stroke:#333
    style ETA fill:#ffcc99,stroke:#333
    style RK fill:#cce5ff,stroke:#333
```

**Routing tile hierarchy:**
- **Level 1 (local):** Small tiles, all streets, for short trips
- **Level 2 (arterial):** Medium tiles, connecting districts
- **Level 3 (highway):** Large tiles, interstates, for cross-country
- Edges connect tiles within and across levels (e.g., freeway entrance links Level 1 to Level 3)

**A* algorithm on routing tiles:**
1. Convert origin/destination to geohash, load start/end routing tiles from S3
2. Expand graph traversal, loading neighboring tiles on demand
3. Jump between tile levels as roads connect (local --> highway)
4. Cache frequently accessed tiles on the routing service

## Deep Dive: Location Service and Kafka Pipeline

```mermaid
graph LR
    Client[Mobile Client<br>batch every 15s]
    LS[Location Service]
    K[Kafka]
    CDB[(Cassandra<br>User Location DB)]
    TDB[(Traffic DB)]
    RTPS[Routing Tile<br>Processing Service]
    ML[ML Service<br>Personalization]
    AN[Analytics Service]

    Client --> LS
    LS --> CDB
    LS --> K
    K --> TDB
    K --> RTPS
    K --> ML
    K --> AN

    style K fill:#ff9999,stroke:#333
    style LS fill:#b3e6b3,stroke:#333
```

- **Cassandra** schema: partition key = `user_id`, clustering key = `timestamp`
- Efficient range queries for a user's location history
- **Kafka** decouples location ingestion from downstream consumers
- Traffic DB updates feed the ETA service for real-time accuracy
- Routing tile processing detects new/closed roads

## Deep Dive: Adaptive ETA and Rerouting

When traffic changes during navigation:
1. Track active navigating users and their routing tiles in a database
2. When a routing tile's traffic changes, find affected users
3. Recalculate ETA and push updates via **WebSocket**
4. Use hierarchical routing tile lookup to quickly filter unaffected users
5. **Delivery protocol:** WebSocket preferred over SSE for bi-directional communication

## Trade-offs

| Decision | Pro | Con |
|---|---|---|
| Pre-generated map tiles | Highly cacheable, CDN-friendly | Storage-intensive (~100 PB) |
| Vector tiles over raster | Better compression, smooth zoom | Requires WebGL on client |
| Routing tiles in S3 | Cheap storage, load on demand | Latency on cache miss |
| 3-level routing hierarchy | Efficient cross-country routing | Complex cross-level edge management |
| Kafka for location stream | Decoupled consumers, replay capability | Additional infrastructure |
| Cassandra for location DB | High write throughput, horizontal scaling | Eventual consistency |
| Batched GPS updates (15s) | 15x QPS reduction | Slightly delayed location data |

## Takeaways

1. **Tile everything** -- both map rendering and routing benefit from spatial partitioning into tiles
2. **CDN is essential** for map tiles -- static content, massive read volume, global distribution
3. **Hierarchical routing tiles** (local/arterial/highway) make continent-scale pathfinding tractable
4. **Kafka streaming** decouples real-time location data from diverse downstream consumers
5. **Batch GPS updates** on the client to reduce server QPS by 15x
6. **ML-based ETA** using live traffic data dramatically improves accuracy over static estimates
7. **Adaptive rerouting** via WebSocket keeps navigation responsive to changing conditions

## Related Concepts

- [[map-tiling]] -- Pre-generated tiles at multiple zoom levels
- [[routing-tiles]] -- Graph partitioning for scalable pathfinding
- [[navigation-service]] -- Route planner with shortest-path, ETA, and ranking
- [[location-service]] -- GPS ingestion via Kafka streaming pipeline
- [[eta-service]] -- ML-based travel time prediction
- [[cdn-map-delivery]] -- CDN strategy for serving map tiles globally
- [[geocoding]] -- Address to coordinate conversion